## 1. Installation

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes datasets
!pip install -q unsloth

## 2. Imports, configuration, reproductibilité

In [ ]:
import json, math, os, random, re, time, hashlib
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DATA_DIR = Path("outputs/step3_dataset")
OUT_DIR = Path("outputs/step4_qlora"); OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"                # base en 4-bit NF4 chargée à la section 4
dataset_cfg = json.load(open(DATA_DIR / "dataset_config.json", encoding="utf-8"))
MAX_SEQ_LENGTH = int(dataset_cfg["stats"]["max_seq_length"])
rag_prompt = json.load(open(DATA_DIR / "rag_prompt.json", encoding="utf-8"))
ABSTAIN_MARKER = rag_prompt["abstain_marker"]

HP = dict(lora_r=16, lora_alpha=16, lora_dropout=0.0, lr=2e-4, epochs=2, per_device_bs=1, grad_accum=16, warmup_ratio=0.05, weight_decay=0.0)
assert torch.cuda.is_available(), "GPU requis (Colab : Exécution > Modifier le type d'exécution > GPU)"
gpu = torch.cuda.get_device_properties(0)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # T4 (7.5) : bf16 seulement émulé -> fp16 ; torch.cuda.is_bf16_supported() peut répondre True à tort
print(f"GPU : {gpu.name} ({gpu.total_memory / 1e9:.0f} Go) | bf16 : {BF16} | max_seq_length : {MAX_SEQ_LENGTH} | k_context étape 3 : {dataset_cfg['k_context']}")

## 3. Données : chargement, vérifications, masquage du prompt

In [ ]:
def load_jsonl(p): return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
train_raw, val_raw = load_jsonl(DATA_DIR / "train.jsonl"), load_jsonl(DATA_DIR / "val.jsonl")
print(f"train : {len(train_raw)} | val : {len(val_raw)}")
print("types train :", dict(Counter(r["meta"]["kind"] for r in train_raw)), "| val :", dict(Counter(r["meta"]["kind"] for r in val_raw)))
assert not ({r["meta"]["id"] for r in train_raw} & {r["meta"]["id"] for r in val_raw}), "id communs entre train et val"
# Le split de l'étape 3 est fait par QUESTION SOURCE : on vérifie qu'aucune question n'est commune.
q_tr = {r["messages"][1]["content"].split("Question :")[-1].strip() for r in train_raw}
q_va = {r["messages"][1]["content"].split("Question :")[-1].strip() for r in val_raw}
assert not (q_tr & q_va), "même question dans train et val"

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.padding_side = "right"
if tok.pad_token is None: tok.pad_token = tok.eos_token

def encode(rec):
    """-> input_ids, labels (-100 sur le prompt), attention_mask."""
    msgs = rec["messages"]
    # On passe par du texte puis on tokenise : indépendant de la version de transformers (le type retourné par apply_chat_template a changé).
    prompt_txt = tok.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True)
    full_txt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    prompt_ids = tok(prompt_txt, add_special_tokens=False)["input_ids"]
    full_ids = tok(full_txt, add_special_tokens=False)["input_ids"]
    assert full_ids[:len(prompt_ids)] == prompt_ids, "le prompt n'est pas un préfixe de l'exemple complet"
    # Qwen ajoute "\n" après <|im_end|> : on retire ce token final de la cible pour ne pas apprendre un token « hors réponse ».
    if tok.decode(full_ids[-1:]) == "\n": full_ids = full_ids[:-1]
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    return dict(input_ids=full_ids, labels=labels, attention_mask=[1] * len(full_ids))

enc_train, enc_val = [encode(r) for r in train_raw], [encode(r) for r in val_raw]
lens = np.array([len(e["input_ids"]) for e in enc_train + enc_val])
n_ans = np.array([sum(l != -100 for l in e["labels"]) for e in enc_train])
print(f"longueur (tokens) : moy {lens.mean():.0f} | p95 {np.percentile(lens, 95):.0f} | max {lens.max()} (limite {MAX_SEQ_LENGTH})")
print(f"tokens appris par exemple : moy {n_ans.mean():.0f} ({n_ans.sum() / sum(len(e['input_ids']) for e in enc_train):.1%} des tokens : le reste est du contexte masqué)")
assert lens.max() <= MAX_SEQ_LENGTH, "exemple plus long que max_seq_length : relancer l'étape 3 (mesure) ou augmenter la limite"

e0 = next(e for e, r in zip(enc_train, train_raw) if r["meta"]["kind"] == "answer")
print("\n--- Partie APPRISE d'un exemple (le reste est masqué) ---")
print(tok.decode([t for t in e0["labels"] if t != -100]))
print("--- Partie masquée (fin du prompt) ---")
print(tok.decode(e0["input_ids"][len(e0["input_ids"]) - len([t for t in e0["labels"] if t != -100]) - 25: len(e0["input_ids"]) - len([t for t in e0["labels"] if t != -100])]))